<a href="https://colab.research.google.com/github/turayusa/homework/blob/main/DX_601_Week_7_Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX 601 Week 7 Homework

## Introduction

In this homework, you will get some practice implementing some models that take advantage of MLE estimates to train using closed-form solutions.

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for sample code.

* https://github.com/bu-cds-omds/dx500-examples
* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Instructions

You should replace every instance of "..." or "TODO" below.
These are where you are expected to write code to answer each problem.

After some of the problems, there are extra code cells that will test functions that you wrote so you can quickly see how they run on an example.
If your code works on these examples, it is more likely to be correct.
However, the autograder will test different examples, so working correctly on these examples does not guarantee full credit for the problem.
You may change the example inputs to further test your functions on your own.
You may also add your own example inputs for problems where we did not provide any.

Be sure to run each code block after you edit it to make sure it runs as expected.
When you are done, we strongly recommend you run all the code from scratch (Runtime menu -> Restart and Run all) to make sure your current code works for all problems.

If your code raises an exception when run from scratch, it will  interfere with the auto-grader process causing you to lose some or all points for this homework.
Please ask for help in YellowDig or schedule an appointment with a learning facilitator if you get stuck.


## Problems

## Shared Imports

Do not install or use any additional modules.
Installing additional modules may result in an autograder failure resulting in zero points for some or all problems.

In [1]:
import math
import os
import random
from abc import abstractmethod, ABC
from tqdm import tqdm # progress bar!
from pprint import pprint
from typing import List, Union

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.special as scsp

### Object Hierarchy

All of the distributions you will fit will have the same api defined by this abstract class (that they will implement) shown below. Please do not modify this:

In [3]:
# please don't touch this
class Distribution(ABC):

    # this is how you make an abstract method in Python
    # all child classes MUST implement this method
    # otherwise we will get an error when the child class
    # is instantiated
    @abstractmethod
    def fit(self,
            X: np.ndarray               # input data to fit from
            ) -> "Distribution":
        ... # same as "pass"

    # another abstract method that every child class will have to implement
    # in order to be able to be instantiated
    @abstractmethod
    def prob(self,
             X: np.ndarray
             ) -> np.ndarray:           # return Pr[x] for each point (row) of the data (X)
        ... # same as "pass"

    @abstractmethod
    def parameters(self) -> List[Union[float, np.ndarray]]:
        ... # same as "pass"

### Problem 1
Please complete the implementation of the 'BinomialDistribution' class. This class has a parameter `p` as well as a value `n` which are stored as fields. Parameter `p` is only set to a meaningful value when the user calls the `fit` method.

In [4]:
class BinomialDistribution(Distribution):
    def __init__(self,
                 n: int) -> None:
        # controlled by parameter "p"
        self.p: float = None
        self.n: int = n

    def fit(self,
            X: np.ndarray                   # input data to fit from. This will be a single row with {n} columns
                                            # each element is an outcome with value 1 = success and 0 = failure
            ) -> "BinomialDistribution":

        if X.shape[-1] != self.n:
            raise ValueError(f"ERROR: expected {n} outcomes in data")
        if X.shape[0] != 1:
            raise ValueError(f"ERROR: expected a single row to fit from")

        # the MLE of p is the fraction of successes to examples (proved in lecture)
        # we need the input to be binary (success or not) where 1 is considered success
        # then its all about counting

        # TODO
        self.p = float(np.mean(X))
        return self # keep this at the end

    def prob(self,
             X: np.ndarray                      # this will be a 2D array where each row has a single scalar in it: the number of successes
             ) -> np.ndarray:

        if X.shape[-1] != 1:
            raise ValueError(f"ERROR: expected 2d data with {1} column")


          # TODO
        k = X[:, 0:1]
        probs = scsp.comb(self.n, k) * (self.p ** k) * ((1 - self.p) ** (self.n - k))

        return probs

    def parameters(self) -> List[Union[float, np.ndarray]]:
        return [self.n, self.p]

In [5]:
# Test it out
np.random.seed(12345) # set random seed for reproducibility

n_binom: int = 10
X_binom: np.ndarray = np.random.choice([0, 1], size=(1,n_binom)) # uniform prob by default

m_binom: Distribution = BinomialDistribution(n_binom).fit(X_binom)
print(m_binom.parameters())

X_test_outcomes = np.random.choice([0,1], size=(4, n_binom))
X_test = np.sum(X_test_outcomes, axis=1, keepdims=True)
print(m_binom.prob(X_test))

[10, 0.5]
[[0.1171875 ]
 [0.24609375]
 [0.00976562]
 [0.20507812]]


### Problem 2

Please complete the implementation of the 'PoissonDistribution' class. This class has a parameter $\lambda$ which is stored as a field called `lam` (`lambda` is a keyword in python). Parameter $\lambda$ is only set to a meaningful value when the user calls the `fit` method.

In [6]:
class PoissonDistribution(Distribution):
    def __init__(self) -> None:
        # controlled by parameter "lambda"
        # but "lambda" is a keyword in python so call it "lamb"
        self.lamb: float = None

    def fit(self,
            X: np.ndarray                   # input data to fit from
                                            # this will be a bunch of integer samples stored in a column vector
            ) -> "PoissonDistribution":

        # the MLE of lambda is the mean of the data
        # TODO
        self.lamb = float(np.mean(X))
        return self # keep this at the end

    def prob(self,
             X: np.ndarray                  # this will be a column vector where every element is an integer
             ) -> np.ndarray:
        if X.shape[-1] != 1:
            raise ValueError("ERROR: expected each example to be a single number")

        # TODO
        k = X[:, 0:1]
        probs = np.exp(scsp.xlogy(k, self.lamb) - self.lamb - scsp.gammaln(k + 1))
        return probs
    def parameters(self) -> List[Union[float, np.ndarray]]:
        return [self.lamb]

In [7]:
# Test it out
X_poisson: np.ndarray = np.array([5, 2, 2, 2, 3, 1, 10]).reshape(-1, 1) # reshape to col vector
m_poisson: Distribution = PoissonDistribution().fit(X_poisson)
print(m_poisson.parameters())


# A tiny experiment:
# Poisson distributions tend to model Basketball (and other goal-based sports) pretty well!

# load in a bunch of steph curry data from 2015-2016 season
steph_curry_df: pd.DataFrame = pd.read_csv("https://raw.githubusercontent.com/catabia/cs391_spring21/main/curry16.csv")

# only look at 3 point column and convert it to a numpy column vector
X_3point: pd.DataFrame = steph_curry_df["3P"].to_numpy().reshape(-1,1) # reshape to col vector
print(X_3point.shape)

m_steph_curry: Distribution = PoissonDistribution().fit(X_3point)
print(m_steph_curry.parameters())

# predict prob of scoring each # of 3pointers
pmf: np.ndarray = m_steph_curry.prob(np.arange(X_3point.max() + 1).reshape(-1, 1))
print(pmf, pmf.sum())

[3.5714285714285716]
(79, 1)
[5.0886075949367084]
[[0.0061666 ]
 [0.03137941]
 [0.07983875]
 [0.13542269]
 [0.17227823]
 [0.17533126]
 [0.14869867]
 [0.1080956 ]
 [0.06875701]
 [0.03887527]
 [0.0197821 ]
 [0.00915121]
 [0.00388058]] 0.9976573768975738


### Problem 3

Please complete the implementation of the 'GaussianDistribution' class. This class has two parameters parameter $\mu$ and $\sigma^2$ which are stored in fields called `mu` and `var`. These parameters are only set to a meaningful value when the user calls the `fit` method.

In [8]:
class GaussianDistribution(Distribution):
    def __init__(self) -> None:
        # controlled by parameters mu and var
        self.mu: float = None
        self.var: float = None

    def fit(self,
            X: np.ndarray                   # input data to fit from
                                            # this will be a bunch of integer samples stored in a column vector
            ) -> "GaussianDistribution":

        # the MLE of mu is the mean of the data while the MLE of var is the var of the data
        # TODO
        self.mu = float(np.mean(X))
        self.var = float(np.var(X, ddof=0))
        return self # keep this at the end

    def prob(self,
             X: np.ndarray                  # this will be a column vector where every element is a float
             ) -> np.ndarray:
        if X.shape[-1] != 1:
            raise ValueError("ERROR: expected each example to be a single number")

        # TODO
        x = X[:, 0:1]
        probs = np.exp(-((x - self.mu) ** 2) / (2 * self.var)) / np.sqrt(2 * np.pi * self.var)

        return probs

    def parameters(self) -> List[Union[float, np.ndarray]]:
        return [self.mu, self.var]

In [9]:
# Test it out
# TODO
X_gaussian = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)
m_gaussian = GaussianDistribution().fit(X_gaussian)

print(m_gaussian.parameters())
print(m_gaussian.prob(X_gaussian))
print("Output shape:", m_gaussian.prob(X_gaussian).shape)

[3.0, 2.0]
[[0.10377687]
 [0.21969564]
 [0.28209479]
 [0.21969564]
 [0.10377687]]
Output shape: (5, 1)
